In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "591ab4e3a3e575d95d805a92550ef076fd65a824"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/591ab4e3a3e575d95d805a92550ef076fd65a824/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Concise Implementation of Linear Regression

In that section we initialized the parameters, defined the
forward pass and squared loss, and implemented the update explicitly. Modern
frameworks provide reusable implementations of each of these components. This
section rebuilds the same model with framework data loaders, layers, losses,
and optimizers, identifying the primitive that replaces each explicit step.

In [ ]:
from d2l import jax as d2l
from flax import nnx
import jax
from jax import numpy as jnp
import optax

## Defining the Model

Each component from that section has a direct counterpart here.
The hand-rolled weight vector $\mathbf{w}$ and bias $b$ are replaced by a single
*layer*; our manual squared-error computation is replaced by a built-in *loss*;
and our explicit parameter-update loop is replaced by an *optimizer* object.
Implementing these components once exposes their behavior. For routine work,
framework components reduce repeated code and provide optimized implementations.

For standard operations,
we can use a framework's predefined layers,
which allow us to focus
on the layers used to construct the model
rather than worrying about their implementation.
Recall the architecture of a single-layer network
as described in the figure.
The layer is called *fully connected*,
since each of its inputs is connected
to each of its outputs
by means of a matrix--vector multiplication.

In [ ]:
class LinearRegression(d2l.Module):
    """The linear regression model implemented with high-level APIs."""
    def __init__(self, num_inputs, lr, rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.net = nnx.Linear(
            num_inputs, 1, kernel_init=nnx.initializers.normal(0.01),
            rngs=rngs)

The `forward` method invokes the predefined layer's built-in `__call__` method.

In [ ]:
@d2l.add_to_class(LinearRegression)
def forward(self, X):
    return self.net(X)

## Defining the Loss Function

In [ ]:
@d2l.add_to_class(LinearRegression)
def loss(self, y_hat, y):
    return d2l.reduce_mean(jnp.square(y_hat - y))

## Defining the Optimization Algorithm

In [ ]:
@d2l.add_to_class(LinearRegression)
def configure_optimizers(self):
    return optax.sgd(self.lr)

## Training

Expressing the model through high-level framework APIs requires fewer lines of
code.
We did not have to allocate parameters individually,
define our loss function, or implement minibatch SGD.
Once we start working with much more complex models,
the advantages of the high-level API will grow considerably.

The batch-level computation is now delegated to standard layers, loss, and
optimizer objects, while the training loop is the same one used from scratch.
We call the `fit` method (introduced in that section),
which relies on the implementation of the `fit_epoch` method
in that section,
to train our model.

In [ ]:
model = LinearRegression(2, lr=0.03)
data = d2l.SyntheticRegressionData(w=d2l.tensor([2, -3.4]), b=4.2)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

Below, we
compare the model parameters learned
by training on finite data
and the actual parameters
that generated our dataset.
This is where the concise version differs conceptually from the scratch one:
the parameters no longer hang off our class as `self.w` and `self.b` but live
*inside* the layer object, so `get_w_b` reaches through `net` to find them.
As in our implementation from scratch,
note that our estimated parameters
are close to their true counterparts.

In [ ]:
@d2l.add_to_class(LinearRegression)
def get_w_b(self):
    return self.net.kernel[...], self.net.bias[...]

w, b = model.get_w_b()

In [ ]:
print(f'error in estimating w: {data.w - d2l.reshape(w, data.w.shape)}')
print(f'error in estimating b: {data.b - b}')

## Summary

MXNet [@Chen.Li.Li.ea.2015], JAX
[@Frostig.Johnson.Leary.2018], PyTorch
[@Paszke.Gross.Massa.ea.2019], and TensorFlow
[@Abadi.Barham.Chen.ea.2016] provide standard implementations of data
loaders, layers, losses, optimizers, and training utilities. These components
are concise, tested, and optimized. Direct implementations remain useful when
a model requires a component that the framework does not provide.


## Exercises

1. **Loss scaling.** The framework loss in `LinearRegression.loss` averages
   over the minibatch and omits the $\frac{1}{2}$ of the equation.
    1. State how the learning rate must change if the average is replaced
       by the sum over the minibatch, and explain why.
    1. Predict how the training curve of `LinearRegression` at `lr=0.03`
       compares with that of `LinearRegressionScratch` at the same learning
       rate, and determine which learning rate makes the two updates
       identical. Verify by training both.
1. [code] **Huber loss.** Replace `LinearRegression.loss` with Huber's
   robust loss

    $$l(y,y') = \begin{cases}|y-y'| -\frac{\delta}{2} & \textrm{ if } |y-y'| > \delta \\ \frac{1}{2 \delta} (y-y')^2 & \textrm{ otherwise,}\end{cases}$$

    with threshold $\delta > 0$. Corrupt one label of
    `SyntheticRegressionData` as in the outlier demonstration of
    that section and train for
    $\delta \in \{0.01, 1, 100\}$. For each $\delta$, state whether the
    recovered $\hat{\mathbf{w}}$ is close to the squared-loss fit, to the
    absolute-error fit, or in between, and relate the pattern to the
    penalty curves in the figure.
1. [code] **Reading gradients.** Compute the gradient of the minibatch
   loss with respect to the weights of `LinearRegression` for a single
   minibatch of `SyntheticRegressionData`, and compare it with the by-hand
   formula the equation. Explain any discrepancy.
1. [code] **Sample-size scaling.** Train `LinearRegression` on
   `SyntheticRegressionData` with `num_train` set to each of
   $5, 10, 20, 50, \ldots, 10{,}000$.
    1. Before running, state how you expect
       $\|\hat{\mathbf{w}} - \mathbf{w}^*\|_2$ and $|\hat{b} - b^*|$ to
       scale with `num_train`.
    1. Plot both errors against `num_train` on logarithmic axes.
    1. Explain why a logarithmically spaced grid is preferable here to
       $1000, 2000, \ldots, 10{,}000$.
1. [code] **Scratch versus concise.** Time `LinearRegressionScratch` and
   `LinearRegression` on the same `SyntheticRegressionData` for 10, 100,
   and 1000 epochs, and report the time per epoch for each. Both run the
   same `fit_epoch`; identify which operations differ between the two runs
   and determine, by timing those operations in isolation, whether they
   account for the measured gap.

[Discussions](https://d2l.discourse.group/t/17977)